In [2]:
#subset by longitude to look for local effects
import os
import sys
import matplotlib
import matplotlib.pyplot as plt
import dask.array as da
import dask.dataframe as dd
import xarray as xr
from xnemogcm import open_domain_cfg, get_metrics
import xgcm
import cartopy.crs as ccrs
import cmocean
import numpy as np
from scipy.stats import linregress
import datetime
import pandas as pd
import plots_spatial as pltspat
# Add SouthernDemons library to PATH
sys.path.append(os.path.abspath("../lib/"))
from teos_ten import teos_sigma0
import datesandtime

# Subdomain information (As inputted into TRACMASS, note non-pythonic indexing)
imindom = 1
imaxdom = 1440
jmindom = 1
jmaxdom = 400
kmindom = 1
kmaxdom = 75

# Location of the TRACMASS run
data_dir = os.path.abspath("/gws/nopw/j04/bas_pog/astyles/ORCA025_fwd/")

# Location of the OUTPUT directory created when running SouthernDemons executable
out_dir = os.path.abspath(data_dir + "/OUTPUT.ORCA025_fwd_extra/")
ndense_path = os.path.abspath("/gws/nopw/j04/bas_pog/astyles/SouthernDemons/neutraldensity/output/ORCA025_Dec1982/*.nc" )
# Location of masks and grid information for the model
grid_path = os.path.abspath("/gws/nopw/j04/bas_pog/astyles/ORCA025_fwd/topo" )
grid_files = ['mask.nc','mesh_hgr.nc','mesh_zgr.nc']

#cal_months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# Use dask to load the tabulated data lazily 
#df_ini = dd.read_parquet(out_dir + f"/df_ini.combined.parquet")
#df_out = dd.read_parquet(out_dir + f"/df_out.combined.parquet")
df_vent = dd.read_parquet(out_dir + f"/df_vent.parquet")
ds_domain = open_domain_cfg( datadir=grid_path, files = grid_files )

In [21]:
ds_nd = xr.open_mfdataset(ndense_path, chunks='auto')
nd_coord = ds_nd.sigma_ver.values

def nd_bin_to_density( x ):
    # If x == -1 -> No density surface intersects the fluid column. Retain value of -1
    if x < 0:
        out = -1

    # Otherwise return the neutral density value
    else:
        out = nd_coord[ x - 1 ]

    return out

In [22]:
df_vent['ndense'] = df_vent['nd_bin_ini'].apply(nd_bin_to_density, meta=('sigma_ver',float))


In [23]:
sum_z = np.array(ds_domain.e3t_1d['gdept_1d'])
z_index = np.array(ds_domain.e3t_1d['z_c'])
cell_z = np.append(np.array(da.diff(ds_domain.e3t_1d['gdept_1d'])),0)
df_sum_in = dd.from_dict({'bin_depth_i':sum_z,'binnedz_i':z_index, 'cell_h_i':cell_z},npartitions =3)
df_merge_in = df_vent.merge(df_sum_in,on = 'binnedz_i')
df_merge_in['depth_i'] = (df_merge_in['bin_depth_i'] - da.floor(df_merge_in['bin_depth_i']))* df_merge_in['cell_h_i'] + df_merge_in['bin_depth_i']

sum_z = np.array(ds_domain.e3t_1d['gdept_1d'])
z_index = np.array(ds_domain.e3t_1d['z_c'])
cell_z = np.append(np.array(da.diff(ds_domain.e3t_1d['gdept_1d'])),0)
df_sum_out = dd.from_dict({'bin_depth_o':sum_z,'binnedz_o':z_index, 'cell_h_o':cell_z},npartitions =3)
df_merge_out = df_vent.merge(df_sum_out,on = 'binnedz_o')
df_merge_out['depth_o'] = (df_merge_out['bin_depth_o'] - da.floor(df_merge_out['bin_depth_o']))* df_merge_out['cell_h_o'] + df_merge_out['bin_depth_o']

cols_to_use = df_merge_out.columns.intersection(df_merge_in.columns)
df_depths = df_merge_out.merge(df_merge_in, on = cols_to_use.to_list() ,how='left')

df_depths = df_depths.drop(columns = ['cell_h_o'])
df_depths = df_depths.drop(columns = ['cell_h_i'])


In [24]:
df_depths.compute().to_parquet("/gws/nopw/j04/bas_pog/evrkin74/Forwards_Ventilation/df_vent.parquet", engine="pyarrow")

In [15]:
df_depths.head(10)

,ntraj_o,x_o,y_o,z_o,subvol_o,time_o,boxface_o,temp_o,sal_o,density_o,...,binnedz_o,nd_bin_ini,sf_zint,bathy_depth_i,bathy_depth_o,ndense,bin_depth_o,depth_o,bin_depth_i,depth_i
0,6844,329.80,369.82,17.34,9.164049e+08,7.460640e+06,0,25.54,35.66,23.54,...,17,0,-140.924212,788.452515,4745.803711,1028.170044,47.211894,48.618604,61.112840,62.005276
1,616817,371.97,167.05,21.00,9.407256e+08,4.903880e+08,6,-1.75,34.04,27.27,...,21,95,11.927550,3230.426514,2948.678467,1027.640966,77.611162,83.306128,508.639904,544.891959
2,695717,38.97,247.00,23.96,9.990873e+08,9.049626e+08,4,1.22,33.89,27.01,...,24,74,-143.428656,4098.403320,4021.936035,1026.970974,108.030281,108.392735,1151.991245,1264.864485
3,701931,63.89,168.03,22.00,9.712372e+08,6.879990e+08,6,-1.72,34.14,27.35,...,22,92,-31.392861,3367.593750,3776.742188,1027.607661,86.929425,96.327670,1265.861417,1370.536987
4,502525,335.00,282.25,13.50,7.170539e+08,4.320000e+03,0,10.31,34.60,26.46,...,13,17,-143.055590,4758.774902,4758.774902,1026.524503,26.558301,28.968073,26.558301,28.968073
5,6732,371.30,356.32,19.62,8.045408e+08,1.080432e+07,0,21.18,35.80,24.92,...,20,2,-139.170450,4723.000977,3040.537109,1025.423833,69.021684,69.207937,61.112840,62.005276
6,538046,12.00,166.28,21.36,9.629624e+08,5.915426e+08,1,-1.34,34.11,27.32,...,21,94,13.002106,2561.199707,3731.000000,1027.631083,77.611162,83.306128,457.625617,489.541042
7,480660,348.85,381.38,15.68,9.203064e+08,3.028320e+06,0,23.44,35.98,24.41,...,16,0,-156.601342,3411.511230,3216.888428,1028.170044,41.180025,42.265910,30.874562,35.129867
8,131596,471.09,91.91,22.96,9.969813e+08,9.253483e+08,0,-1.30,34.21,27.39,...,23,108,17.453036,2789.053223,3906.644287,1027.748382,97.041313,97.495295,1045.854302,1136.527291
9,4567,347.78,300.96,16.13,7.614913e+08,2.596320e+06,0,12.63,34.63,26.06,...,16,12,-142.371400,5113.102051,5174.177246,1026.152465,41.180025,42.265910,41.180025,42.265910


In [7]:
df_depths = df_depths.drop(columns = ['cell_h_o'])
df_depths = df_depths.drop(columns = ['cell_h_i'])

,ntraj_o,x_o,y_o,z_o,subvol_o,time_o,boxface_o,temp_o,sal_o,density_o,density10_o,ntrajc,ntraj_i,x_i,y_i,z_i,subvol_i,time_i,boxface_i,temp_i,sal_i,density_i,density10_i,year_i,month_i,day_i,dayofyear_i,year_o,month_o,day_o,dayofyear_o,binnedx_i,binnedy_i,binnedz_i,binnedx_o,binnedy_o,binnedz_o,nd_bin_ini,sf_zint,bathy_depth_i,bathy_depth_o,bin_depth_o,depth_o,bin_depth_i,cell_h_i,depth_i
npartitions=236,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,int64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,int64,int64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [ ]:
####ad  weddel_bool

In [3]:
data_dir = os.path.abspath("/gws/nopw/j04/bas_pog/evrkin74/Forwards_Ventilation")
df_vent = dd.read_parquet(data_dir + f"/df_vent.parquet")


In [4]:
len(df_vent)

54157552

In [7]:
df_gyre = df_vent[(df_vent['sf_zint']<200) & (df_vent['sf_zint']>10)]
df_weddel_gyre = df_gyre[(df_gyre['binnedx_i']>930)]
df_group = df_weddel_gyre[['binnedx_i','binnedy_i','subvol_i']].groupby(['binnedx_i','binnedy_i'])
df_gyre_copy = df_group.max('subvol_i').compute()
df_gyre_copy=df_gyre_copy.reset_index()
df_gyre_copy=df_gyre_copy[['binnedx_i','binnedy_i']]
### merge bool onto original data frame (1 if row in gyre)
#df_gyre_copy = df_weddel_gyre.copy()[['binnedx_i','binnedy_i']]
df_gyre_copy = df_gyre_copy.assign(weddel_bool=1) # 0 ventilates not in gyre, 1 ventilates in gyre
df_gyre_copy=df_gyre_copy.rename(columns={"binnedx_i": "binnedx_o", "binnedy_i": "binnedy_o"})  
df_merge = df_vent.merge(df_gyre_copy,on = ['binnedx_o','binnedy_o'],how = 'left')



In [10]:
df_merge["weddel_bool"] = df_merge["weddel_bool"].fillna(0)


In [12]:
df_merge.compute().to_parquet("/gws/nopw/j04/bas_pog/evrkin74/Forwards_Ventilation/df_vent.parquet", engine="pyarrow")

In [11]:
df_merge.dtypes

ntraj_o            int64
x_o              float64
y_o              float64
z_o              float64
subvol_o         float64
time_o           float64
boxface_o          int64
temp_o           float64
sal_o            float64
density_o        float64
density10_o      float64
ntrajc             int64
ntraj_i            int64
x_i              float64
y_i              float64
z_i              float64
subvol_i         float64
time_i           float64
boxface_i          int64
temp_i           float64
sal_i            float64
density_i        float64
density10_i      float64
year_i             int64
month_i            int64
day_i              int64
dayofyear_i        int64
year_o             int64
month_o            int64
day_o              int64
dayofyear_o        int64
binnedx_i          int64
binnedy_i          int64
binnedz_i          int64
binnedx_o          int64
binnedy_o          int64
binnedz_o          int64
nd_bin_ini         int64
sf_zint          float64
bathy_depth_i    float64


In [13]:
len(df_merge)

54157552